In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

from polsarpro.io import open_netcdf_beam
from polsarpro.util import pauli_rgb

# change to your local C-PolSARpro install dir
# Kept for consistency with the other dev notebooks, but unused in this
# UAVSAR-only variant.
c_psp_dir = "/home/c_psp/Soft/bin/"
os.environ["PATH"] += os.pathsep + f"{c_psp_dir}/data_process_sngl/"
os.environ["PATH"] += os.pathsep + f"{c_psp_dir}/data_convert/"

input_uavsar_nc = Path("/data/psp/test_files/blackw_20801_22010_006_220507_L090_CX_02_ML5X5.nc")
incidence_angle_file = Path(
    "/data/psp/test_files/blackw_20801_22010_006_220507_L090_CX_02.inc"
)
file_out = Path("/data/psp/res/test_dubois_surface_inversion_uavsar.nc")

with xr.open_dataset(input_uavsar_nc) as raw_ds:
    center_wavelength_cm = raw_ds["metadata"].attrs["Annotation:CenterWavelength"]

# Dubois expects GHz; UAVSAR metadata stores the center wavelength in cm.
freq_ghz = 29.9792458 / float(center_wavelength_cm)

# Leave thresholds as-is for now.
thresh1_db = 0.0
thresh2_db = -10.0
calibration_coeff = 1.0

print(f"Derived radar frequency: {freq_ghz:.6f} GHz")


## Load UAVSAR data and incidence angle raster

In [ ]:
C3 = open_netcdf_beam(input_uavsar_nc)
row_dim, col_dim = C3.dims
# nlines = C3.sizes[row_dim]
# ncols = C3.sizes[col_dim]

# use lines from .ann file
# inc.set_rows                                      (pixels)    = 12344                  ; INC Lines
# inc.set_cols                                      (pixels)    = 11248                  ; INC Samples

nlines = 12344
ncols = 11248

# The incidence-angle file is a little-endian float32 stream in radians.
# Read only the raster-sized prefix and reshape it to the UAVSAR grid.
incidence_angle_values = np.fromfile(
    incidence_angle_file,
    dtype="<f4",
    count=nlines * ncols,
)

# reduce to the same dimensions as multilooked 5x5 image
incidence_angle_values[incidence_angle_values==-10000] = np.nan
incidence_angle = xr.DataArray(
    incidence_angle_values.reshape(nlines, ncols),
    dims=(row_dim, col_dim),
    # coords={row_dim: C3.coords[row_dim], col_dim: C3.coords[col_dim]},
    name="incidence_angle",
).coarsen(y=5, x=5, boundary="trim").mean()


In [ ]:
# rgb = pauli_rgb(C3)

In [ ]:
# rgb[:,::4,1000:].plot.imshow()
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
plt.imshow(rgb.transpose("y", "x", "band"), interpolation="nearest")
# plt.colorbar(fraction=0.046, pad=0.04)

In [ ]:
# rgb[:,::4,1000:].plot.imshow()
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
# plt.imshow(incidence_angle[::5, ::5], interpolation="bilinear")
plt.imshow(incidence_angle, interpolation="nearest")
plt.colorbar(fraction=0.046, pad=0.04)